In [1]:

%load_ext autoreload
%autoreload 2
from data.game_infos import get_game_info
from data.league_infos import get_league_urls, get_game_urls
import pandas as pd
from app.db import create_db_engine
from tqdm import tqdm
import uuid
engine = create_db_engine()
root = 'https://www.handball.net/ligen?organization=Hamburg'
league_urls = get_league_urls(root)

In [ ]:
current_status = pd.read_sql_table('game_details', engine)
hamburg_data = pd.DataFrame()
for league_url in tqdm(league_urls, position=0, leave=True):
    games_urls = get_game_urls(league_url)
    game_data_list = []
    
    
    game_ids = current_status['game_id'].unique()
    games_urls_filtered = [game_url for game_url in games_urls if game_url.split('/')[-1] not in game_ids]
    for game_url in tqdm(games_urls_filtered, position=0, leave=True):
        game_info = get_game_info(game_url)
        game_data_list.append(game_info)
        if game_info.empty:
            break

    hamburg_data = pd.concat([hamburg_data, pd.concat(game_data_list)], ignore_index=True)

In [ ]:
# Create league table
hamburg_data[["league_id", "league_name"]].drop_duplicates().to_sql('leagues', engine, if_exists='replace', index=False)
# Create Team Table
uuid_mapping = {team: str(uuid.uuid4()) for team in hamburg_data['team'].unique()}
hamburg_data['team_id'] = hamburg_data['team'].map(uuid_mapping)
hamburg_data["team_name"] = hamburg_data["team"]
hamburg_data[["team_id", "team_name","league_id"]].drop_duplicates().to_sql('teams', engine, if_exists='replace', index=False)

# Create Game Table
hamburg_data["record_id"] = hamburg_data["game_id"].apply(lambda x: uuid.uuid4())
hamburg_data = hamburg_data.drop(columns=["league_name", "team_name", "team"])
hamburg_data.to_sql('game_details', engine, if_exists='replace', index=False)